In [ ]:
from utils_extraction import tokenize
from utils_extraction import extract_few_shot_examples_from_labels
from utils_extraction import select_few_shot 
from utils_extraction import process_labels
from models import GPTAssistant

In [88]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 20  # Number of few-shot examples to use

#### Define the text to process, and where to save it

In [128]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2019SCC65"
anno = "llm"
version = "v1.2"
out_version = "v1.3"
html_path = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\{filename}_llm_{version}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"


# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\2019SCC65_llm_v1.2.html
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [130]:
# ---------- Tokenize html content ----------
tokens = tokenize(html_content)

In [131]:
# ---------- Tokenize html content ----------
fs_tokens = tokenize(fs_html_content)

In [132]:
if out_version == "v1.1":
    sublabel_config = {
    "parent":["decision", "legislation", "secondary sources"], # only extract sublabels under these parents
    "already_labeled":[], # do not extract sublabels under these labels
    "new_labels":["title", "fragment"],
    "keep_attributes":["labelname"],
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]
    
if out_version == "v1.2":
    sublabel_config = {
    "parent":["secondary sources"], # only extract sublabels under these parents
    "already_labeled":["title", "fragment"], # do not extract sublabels under these labels
    "new_labels":["source", "authors"],
    "keep_attributes":["labelname"], 
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]

if out_version == "v1.3":
    sublabel_config = {
        "parent":["decision", "legislation"], # only extract sublabels under these parents
        "already_labeled":["title", "fragment", "source", "authors"], # do not extract sublabels under these labels
        "new_labels":["citation"],
        "keep_attributes":["labelname"], 
        "switch_type":True, # manual_label -> auto_label
        "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.8]

# Do not use remove_labels here, as we need the parent labels to identify sublabels : This could  create issues.

#### Get few shot

In [137]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples_from_labels(fs_tokens, 
                                              sublabel_config)


# Select examples with distributed method: 50% with "source" label, 50% random others
selected_few_shot_examples = select_few_shot(
    examples=few_shot_examples, 
    n=n_few_shot,
    method="distributed",
    list_of_labels=sublabel_config["new_labels"],
    distribution=distribution
)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")


   ✓ Selected 20 few-shot examples for processing.


In [138]:
selected_few_shot_examples

[('<decision><title>Canadian Pacific Ltd. v. Canada (Competition Act, Director of Investigation and Research)</title>, [1995] O.J. No. 4148 (Gen. Div.)</decision>',
  '<decision><title>Canadian Pacific Ltd. v. Canada (Competition Act, Director of Investigation and Research)</title>, <citation>[1995] O.J. No. 4148 (Gen. Div.)</citation></decision>'),
 ('<decision><title>Alcan-Colony Contracting Ltd. v. M.N.R.</title>, [1971] 2 O.R. 365, 18 D.L.R. (3d) 32, 71 D.T.C. 5082 (H.C.J.)</decision>',
  '<decision><title>Alcan-Colony Contracting Ltd. v. M.N.R.</title>, <citation>[1971] 2 O.R. 365</citation>, <citation>18 D.L.R. (3d) 32</citation>, <citation>71 D.T.C. 5082 (H.C.J.)</citation></decision>'),
 ('<decision><title>Anderson Exploration Ltd. v. Pan-Alberta Gas Ltd. (1998)</title>, 61 Alta. L.R. (3d) 38, [1998] 10 W.W.R. 633 (Q.B.)</decision>',
  '<decision><title>Anderson Exploration Ltd. v. Pan-Alberta Gas Ltd. (1998)</title>, <citation>61 Alta. L.R. (3d) 38</citation>, <citation>[1998]

#### Processing

In [139]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [142]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_sublabels_extraction_from_parent_cot.txt"

processed_label = process_labels(
    model=model,
    tokens=tokens,
    sublabel_config=sublabel_config,
    few_shot_examples=few_shot_examples,
    prompt_path=prompt_path,
    output_dir=output_dir,
    filename=filename,
    max_fallback_attempts=1
)


   ✓ Found 1254 parent mentions to process
   ✓ Built 2498 token segments (1254 to process)


Processing mentions: 100%|██████████| 2498/2498 [1:01:32<00:00,  1.48s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\history_2019SCC65_sublabel.json
   ✓ Processed tokens saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\processed_sublabels_2019SCC65.json

   ✓ Sublabel extraction completed:
      - Total mentions: 1254
      - Successful: 1254
      - Failed: 0


## Post Processing

In [ ]:
# Useless verification to check if the tokens are the same after processing (except for the auto labels)
t1 = []
t2 = []
for token in processed_label:
    if not is_auto_label_tag(token) in [1, 2]:
        t1.append(token)


for token in tokens:
    if not is_auto_label_tag(token) in [1, 2]:
        t2.append(token)

assert t1 == t2, "The tokens are different after processing, which should not happen as we are only adding auto_label tags without changing the original tokens."


processed_html = decode(processed_label)

print(f"\HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = add_style_and_parent_to_auto_labels(processed_html)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_{anno}_{out_version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")



Merged HTML length: 845139
